In this work, we use transformer model to integrate gene expression and TCR amino acid sequences

Getting gene data

In [1]:
# %matplotlib inline

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os 
import numpy as np

import pandas as pd
# import seaborn as sb
import matplotlib.pyplot as pl

import scanpy as sc

import anndata as ad

from scipy.sparse import csr_matrix
from matplotlib import rcParams
from matplotlib import colors

sc.settings.verbosity = 3


In [2]:
gene_TCR = ad.read_h5ad('../10Xdatasets/gex_merge_log1_5000_genes_all_peptides.h5ad')
gene_TCR

c:\Users\phill\anaconda3\envs\tensorflow\lib\site-packages\anndata\_core\anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 145479 × 5000
    obs: 'v_gene_TRA', 'v_gene_TRB', 'd_gene_TRA', 'd_gene_TRB', 'j_gene_TRA', 'j_gene_TRB', 'c_gene_TRA', 'c_gene_TRB', 'cdr3_TRA', 'cdr3_TRB', 'cdr3_nt_TRA', 'cdr3_nt_TRB', 'umis_TRA', 'umis_TRB', 'donor', 'cell_clono_cdr3_aa', 'cell_clono_cdr3_nt', 'CD3', 'CD19', 'CD45RA', 'CD4', 'CD8a', 'CD14', 'CD45RO', 'CD279_PD-1', 'IgG1', 'IgG2a', 'IgG2b', 'CD127', 'CD197_CCR7', 'HLA-DR', 'A0101_VTEHDTLLY_IE-1_CMV', 'A0201_KTWGQYWQV_gp100_Cancer', 'A0201_ELAGIGILTV_MART-1_Cancer', 'A0201_CLLWSFQTSA_Tyrosinase_Cancer', 'A0201_IMDQVPFSV_gp100_Cancer', 'A0201_SLLMWITQV_NY-ESO-1_Cancer', 'A0201_KVAELVHFL_MAGE-A3_Cancer', 'A0201_KVLEYVIKV_MAGE-A1_Cancer', 'A0201_CLLGTYTQDV_Kanamycin-B-dioxygenase', 'A0201_LLDFVRFMGV_EBNA-3B_EBV', 'A0201_LLMGTLGIVC_HPV-16E7_82-91', 'A0201_CLGGLLTMV_LMP-2A_EBV', 'A0201_YLLEMLWRL_LMP1_EBV', 'A0201_FLYALALLL_LMP2A_EBV', 'A0201_GILGFVFTL_Flu-MP_Influenza', 'A0201_GLCTLVAML_BMLF1_EBV', 'A0201_NLVPMVATV_pp65_CMV', 'A0201_I

In [3]:
gene_TCR = gene_TCR[gene_TCR.obs.donor == 'donor2']
gene_TCR

View of AnnData object with n_obs × n_vars = 58337 × 5000
    obs: 'v_gene_TRA', 'v_gene_TRB', 'd_gene_TRA', 'd_gene_TRB', 'j_gene_TRA', 'j_gene_TRB', 'c_gene_TRA', 'c_gene_TRB', 'cdr3_TRA', 'cdr3_TRB', 'cdr3_nt_TRA', 'cdr3_nt_TRB', 'umis_TRA', 'umis_TRB', 'donor', 'cell_clono_cdr3_aa', 'cell_clono_cdr3_nt', 'CD3', 'CD19', 'CD45RA', 'CD4', 'CD8a', 'CD14', 'CD45RO', 'CD279_PD-1', 'IgG1', 'IgG2a', 'IgG2b', 'CD127', 'CD197_CCR7', 'HLA-DR', 'A0101_VTEHDTLLY_IE-1_CMV', 'A0201_KTWGQYWQV_gp100_Cancer', 'A0201_ELAGIGILTV_MART-1_Cancer', 'A0201_CLLWSFQTSA_Tyrosinase_Cancer', 'A0201_IMDQVPFSV_gp100_Cancer', 'A0201_SLLMWITQV_NY-ESO-1_Cancer', 'A0201_KVAELVHFL_MAGE-A3_Cancer', 'A0201_KVLEYVIKV_MAGE-A1_Cancer', 'A0201_CLLGTYTQDV_Kanamycin-B-dioxygenase', 'A0201_LLDFVRFMGV_EBNA-3B_EBV', 'A0201_LLMGTLGIVC_HPV-16E7_82-91', 'A0201_CLGGLLTMV_LMP-2A_EBV', 'A0201_YLLEMLWRL_LMP1_EBV', 'A0201_FLYALALLL_LMP2A_EBV', 'A0201_GILGFVFTL_Flu-MP_Influenza', 'A0201_GLCTLVAML_BMLF1_EBV', 'A0201_NLVPMVATV_pp65_CMV', '

In [4]:
gene_TCR.obs.donor

barcode
CTGCTGTAGCGAAGGG-32    donor2
AAGGCAGAGAGTAATC-36    donor2
CGTGTCTCAGGGCATA-30    donor2
CGTGTAAAGGGCTTCC-5     donor2
CCCTCCTAGGGTTTCT-22    donor2
                        ...  
GATCAGTAGCCTTGAT-6     donor2
GGTATTGGTCTGGTCG-3     donor2
TCAGATGCACAGTCGC-36    donor2
AGCGTATGTAAGAGGA-8     donor2
CCACTACGTGCAACGA-40    donor2
Name: donor, Length: 58337, dtype: category
Categories (1, object): ['donor2']

In [5]:
gene = pd.DataFrame(gene_TCR.X.todense())
gene

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,1.098612,0.693147,0.0,0.0
1,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.693147,0.0,0.000000,0.0,0.000000,2.302585,0.0,0.0
2,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0
3,0.0,0.000000,0.0,0.0,0.0,0.693147,0.000000,0.693147,0.693147,0.0,...,0.0,0.000000,0.000000,0.0,0.693147,0.0,1.386294,0.693147,0.0,0.0
4,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.0,1.098612,0.0,0.000000,1.386294,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58332,0.0,0.693147,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.693147,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,1.791759,1.386294,0.0,0.0
58333,0.0,0.000000,0.0,0.0,0.0,0.000000,0.693147,0.000000,0.000000,0.0,...,0.0,0.693147,0.000000,0.0,0.000000,0.0,0.693147,1.098612,0.0,0.0
58334,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,1.098612,0.693147,0.0,0.0
58335,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.693147,0.000000,0.0,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,1.098612,1.098612,0.0,0.0


In [6]:
tcr_seq = gene_TCR.obs[['cdr3_TRB']]
tcr_seq

,cdr3_TRB
barcode,
CTGCTGTAGCGAAGGG-32,CASSLRDGSEAFF
AAGGCAGAGAGTAATC-36,CSAKEGPTGGYGYTF
CGTGTCTCAGGGCATA-30,CASSSGLAGVNEQFF
CGTGTAAAGGGCTTCC-5,CASSQRPSEVGELFF
CCCTCCTAGGGTTTCT-22,CSAGSGTRGETQYF
...,...
GATCAGTAGCCTTGAT-6,CASSPVTGGGSGANVLTF
GGTATTGGTCTGGTCG-3,CASSSTGGGEKDQPQHF
TCAGATGCACAGTCGC-36,CASSQGQKDTDTQYF


In [7]:
import tensorflow as tf

In [8]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Reshape, Conv2D, Conv2DTranspose, Flatten, Dropout
from tensorflow.keras.initializers import HeNormal

# Define input layer
input_gex = Input(shape=(100,))
gex = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(input_gex)
gex = Reshape(target_shape=(8,8,1))(gex)

# Convolutional layers
gex = Conv2D(filters=64, kernel_size=3, strides=1, activation='relu', padding="same")(gex)
gex = Conv2D(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(gex)

gex = Flatten()(gex)
hidden_layer = Dense(units=225, activation='relu', kernel_initializer=HeNormal())(gex)

# Transposed Convolutional layers
tcr = Reshape(target_shape=(15,15,1))(hidden_layer)
tcr = Conv2DTranspose(filters=16, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
tcr = Conv2DTranspose(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
tcr = Dropout(rate=0.2)(tcr)
tcr = Flatten()(tcr)
tcr = Dense(units=150, activation='relu', kernel_initializer=HeNormal())(tcr)
tcr = Dense(units=130, activation='relu')(tcr)  # Ensure proper activation
tcr = Dense(units=121, activation='relu')(tcr)
# Define model
model = Model(inputs=input_gex, outputs=tcr)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0015),
              loss='mse')

# Check layer names
model.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 100)]             0         
                                                                 
 dense (Dense)               (None, 64)                6464      
                                                                 
 reshape (Reshape)           (None, 8, 8, 1)           0         
                                                                 
 conv2d (Conv2D)             (None, 8, 8, 64)          640       
                                                                 
 conv2d_1 (Conv2D)           (None, 8, 8, 32)          18464     
                                                                 
 flatten (Flatten)           (None, 2048)              0         
                                                                 
 dense_1 (Dense)             (None, 225)               461025

In [22]:
AE_tcr = pd.read_csv("../AE_emb_TRA_all_peptides_10X_donor_2_only.csv")
AE_tcr

,0,1,2,3,4,5,6,7,8,9,...,111,112,113,114,115,116,117,118,119,120
0,76.04199,-288.03503,-319.26965,-532.712040,208.536060,-64.99309,-85.555730,-199.236370,410.111820,286.18753,...,-432.441830,-107.840800,-240.652050,-406.67280,-192.62181,-194.217910,-257.94235,774.336700,497.803130,-1537.46310
1,-356.04398,-342.22836,169.23965,16.565151,-595.301000,-298.07706,-359.217600,-607.446100,-623.646670,822.46625,...,-262.469420,378.462700,-581.311040,-630.82904,247.68658,-692.218300,-314.18420,1051.592700,-78.261250,-990.46250
2,-585.36670,295.48532,-751.05290,-381.678530,-847.026600,-1345.90930,14.967749,-495.210400,-106.591390,-335.15250,...,144.986970,950.169070,209.585190,-240.45667,481.21350,55.207410,604.74170,1995.057300,-830.159060,-785.85187
3,361.37590,-1815.36740,-379.34860,104.582600,458.532800,-731.26740,-315.524320,353.166930,-17.143927,255.07234,...,215.707020,-528.280300,206.634740,-147.23466,-473.15222,949.036740,-926.45886,869.648440,34.920956,-204.32886
4,-670.10156,326.64026,735.83514,-13.041043,-82.202095,992.32910,114.262170,-775.427060,-288.377400,763.08270,...,-495.978100,74.772860,1021.259400,-238.74141,-1030.06570,-202.258620,211.66460,-61.895233,-182.688430,-584.59930
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58332,-642.90247,140.76115,-241.00450,-706.352500,52.436300,-737.55910,-1197.096600,-1325.599600,-634.458440,156.51491,...,49.532700,602.190300,-23.398672,13.02500,-896.68600,-697.707760,7.86235,617.627700,340.211580,-328.09286
58333,374.96194,-954.93600,759.42847,-500.034400,-14.056041,-454.97516,103.059760,-693.350770,-327.388580,-244.72058,...,-135.847090,108.074104,-232.036480,-286.90018,-489.43332,51.709730,-1052.57960,1277.031700,-266.550380,-193.93994
58334,-1082.19760,447.98386,194.00418,541.659100,-365.770750,129.08498,186.959300,-52.338303,444.221680,-525.80750,...,-238.630280,691.062400,313.952580,471.24512,573.11707,-85.426125,-132.69270,1013.949340,-821.222000,901.69354
58335,-558.54083,293.67313,-198.75066,883.469500,-687.101500,235.74162,-523.161800,-64.029340,324.893130,385.24008,...,-315.246830,-68.296790,539.368500,77.00570,-98.10768,1201.100200,-499.11572,564.900630,-573.257900,-705.76350


In [11]:
import numpy as np
from sklearn.decomposition import NMF

# Generate random non-negative data
data = gene.to_numpy()

# Initialize the NMF model
n_components = 100
model_nmf = NMF(n_components=n_components, init='random', random_state=0)

# Fit the model to the data
W = model_nmf.fit_transform(data)
H = model_nmf.components_

# Display the results
print("Basis matrix (W):\n", W)
print("Coefficients matrix (H):\n", H)


c:\Users\phill\anaconda3\envs\tensorflow\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


Basis matrix (W):
 [[0.378935   0.         0.00947428 ... 0.         0.02493701 0.01647122]
 [0.         0.00638552 0.         ... 0.00840525 0.         0.01527937]
 [0.7843851  0.         0.         ... 0.0081527  0.02149329 0.        ]
 ...
 [0.14777295 0.01624463 0.         ... 0.         0.06578172 0.01559841]
 [0.         0.00468671 0.09291022 ... 0.         0.03712089 0.02740654]
 [0.507081   0.011732   0.09511811 ... 0.         0.         0.00134601]]
Coefficients matrix (H):
 [[0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 0.0000000e+00]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 0.0000000e+00]
 [0.0000000e+00 1.9500397e-01 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 1.5737361e-04]
 ...
 [0.0000000e+00 1.5654299e-01 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 1.3000196e-04]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 0.0000000e+00]
 [0.0000000e+00 5.4106718e-01 0.0000000e+

In [12]:
W = pd.DataFrame(W)
W

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.378935,0.000000,0.009474,0.080753,0.320173,0.002102,0.044627,0.025178,0.045870,0.000000,...,0.041698,0.000000,0.000748,6.299272e-05,0.004852,0.005017,0.023440,0.000000,0.024937,0.016471
1,0.000000,0.006386,0.000000,0.000000,0.271537,0.004310,0.040686,0.006207,0.023888,0.000000,...,0.000212,0.000000,0.021225,8.068039e-03,0.004837,0.004123,0.017595,0.008405,0.000000,0.015279
2,0.784385,0.000000,0.000000,0.076092,0.365910,0.002138,0.040690,0.019331,0.330639,0.068402,...,0.001149,0.083123,0.062468,2.246008e-02,0.012396,0.003521,0.013797,0.008153,0.021493,0.000000
3,0.815916,0.155018,0.064627,0.157953,0.048367,0.002059,0.034770,0.078378,0.421686,0.059805,...,0.000605,0.042623,0.051838,2.371532e-07,0.000240,0.005242,0.019074,0.000000,0.036995,0.027922
4,0.703378,0.000000,0.000000,0.056914,0.553531,0.008509,0.026223,0.056163,0.494144,0.132862,...,0.000000,0.000000,0.058255,1.240427e-02,0.005109,0.002210,0.019312,0.000000,0.000000,0.013547
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58332,0.889756,0.251973,0.155098,0.137700,0.364955,0.002323,0.005447,0.025416,0.366555,0.073269,...,0.000000,0.035508,0.046572,8.042403e-03,0.000926,0.006646,0.014357,0.000000,0.025614,0.017641
58333,1.074156,0.000000,0.298710,0.171526,0.278849,0.004978,0.017507,0.040516,0.162174,0.000000,...,0.003040,0.000500,0.000351,1.742328e-02,0.006008,0.006713,0.017989,0.000000,0.039763,0.010514
58334,0.147773,0.016245,0.000000,0.000000,0.447578,0.003138,0.041926,0.042767,0.225851,0.032842,...,0.000000,0.000000,0.034075,0.000000e+00,0.008273,0.005716,0.007509,0.000000,0.065782,0.015598
58335,0.000000,0.004687,0.092910,0.000000,0.057843,0.001938,0.015773,0.057381,0.000000,0.000000,...,0.000000,0.066383,0.035002,0.000000e+00,0.004015,0.003190,0.014518,0.000000,0.037121,0.027407


In [23]:

# es_callback = EarlyStopping(monitor= 'val_auc', patience=20, restore_best_weights=True)
reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=50, monitor='loss', min_delta=100)

history = model.fit(W,AE_tcr, 
                epochs=1400, 
                batch_size=128, 
                shuffle = True,
                # callbacks=[es_callback, checkpoint,reduce_learning_rate])
                # callbacks=[reduce_learning_rate]
                )

Epoch 1/1400
456/456 [==============================] - 5s 12ms/step - loss: 237442.6719
Epoch 2/1400
456/456 [==============================] - 5s 12ms/step - loss: 223421.0469
Epoch 3/1400
456/456 [==============================] - 5s 12ms/step - loss: 219380.0000
Epoch 4/1400
456/456 [==============================] - 5s 12ms/step - loss: 216766.0625
Epoch 5/1400
456/456 [==============================] - 5s 12ms/step - loss: 214169.0938
Epoch 6/1400
456/456 [==============================] - 5s 12ms/step - loss: 211925.5781
Epoch 7/1400
456/456 [==============================] - 5s 12ms/step - loss: 210324.8750
Epoch 8/1400
456/456 [==============================] - 5s 12ms/step - loss: 208733.6406
Epoch 9/1400
456/456 [==============================] - 6s 12ms/step - loss: 207401.1562
Epoch 10/1400
456/456 [==============================] - 6s 12ms/step - loss: 206173.7500
Epoch 11/1400
456/456 [==============================] - 6s 13ms/step - loss: 204954.8125
Epoch 12/1400
456/4

In [24]:
for layer in model.layers:
    print(layer.name)

input_1
dense
reshape
conv2d
conv2d_1
flatten
dense_1
reshape_1
conv2d_transpose
conv2d_transpose_1
dropout
flatten_1
dense_2
dense_3
dense_4


In [25]:
from tensorflow.keras.models import Model
latent_model = Model(inputs=input_gex, outputs=hidden_layer)


In [ ]:
model.predict( W.iloc[1:2])

In [ ]:
model.predict( W.iloc[4:5])

In [26]:
integration_pred = latent_model.predict( W)

1824/1824 [==============================] - 3s 1ms/step


In [ ]:
pd.DataFrame(integration_pred[1:50,1:50])

In [27]:
integration_pred = integration_pred.reshape([integration_pred.shape[0],-1])

In [28]:
integration_pred.shape

(58337, 225)

In [29]:
integration_pred = integration_pred.reshape([integration_pred.shape[0],-1])
integration_pred.shape

(58337, 225)

In [30]:
pd.DataFrame(integration_pred)

,0,1,2,3,4,5,6,7,8,9,...,215,216,217,218,219,220,221,222,223,224
0,350.240295,0.0,296.866150,0.0,0.0,74.445808,0.000000,33.840343,285.250458,0.0,...,304.450134,0.000000,374.433655,0.000000,0.0,169.616302,0.0,195.215866,0.000000,0.0
1,11.709624,0.0,39.170998,0.0,0.0,0.000000,10.562683,0.000000,0.000000,0.0,...,174.081909,85.569107,232.845901,378.418823,0.0,306.457672,0.0,382.409241,411.157532,0.0
2,259.285706,0.0,244.539871,0.0,0.0,0.000000,4.575391,39.331039,288.906006,0.0,...,0.000000,345.757416,44.337204,207.343445,0.0,264.496918,0.0,0.000000,95.640465,0.0
3,59.300335,0.0,223.926102,0.0,0.0,363.936829,230.102264,92.580536,52.359249,0.0,...,246.660919,0.000000,34.284458,65.706978,0.0,101.435585,0.0,0.000000,294.386078,0.0
4,205.349716,0.0,73.656342,0.0,0.0,0.000000,33.966270,0.000000,0.000000,0.0,...,245.423630,116.727379,100.532776,35.331985,0.0,3.337259,0.0,529.072815,134.370392,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58332,140.859497,0.0,280.412415,0.0,0.0,116.357101,0.000000,31.353722,221.272690,0.0,...,0.000000,0.000000,133.396408,219.419754,0.0,467.866638,0.0,223.181229,303.860260,0.0
58333,174.614578,0.0,159.098221,0.0,0.0,152.347748,323.938263,121.887787,222.605988,0.0,...,0.000000,234.135406,154.922546,100.378105,0.0,255.113922,0.0,0.000000,157.153152,0.0
58334,69.434120,0.0,131.344971,0.0,0.0,0.000000,172.322769,90.610283,79.504738,0.0,...,95.636871,234.803513,146.430222,125.182716,0.0,156.248398,0.0,70.414772,198.899460,0.0
58335,129.113113,0.0,61.013752,0.0,0.0,0.000000,463.973724,209.235733,0.000000,0.0,...,14.921125,0.000000,78.717216,0.000000,0.0,139.688202,0.0,0.000000,27.755493,0.0


In [31]:
pd.DataFrame(integration_pred).to_csv("integration_pred_new_method_gex_to_TCR_alpha_chain_10X_donor_2_only.csv", index=False)